# Module 04 — Notebook 02: Feature Ablation on Tabular Data

## Learning Objectives
By completing this notebook, you will:
1. Train a compact MLP on the UCI Wine Quality dataset
2. Apply `FeatureAblation` to explain individual wine quality predictions
3. Compute global feature importance by aggregating local attributions
4. Compare Feature Ablation with Integrated Gradients for the same predictions

## Prerequisites
- Module 01 Notebook 02 (gradient methods on tabular data)
- Guide 01 (Feature Ablation theory)

## Estimated Time: 15 minutes

---

In [ ]:
learning_objectives(["— Notebook 02: Feature Ablation on Tabular Data", "Train a compact MLP on the UCI Wine Quality dataset", "Apply `FeatureAblation` to explain individual wine quality predictions", "Compute global feature importance by aggregating local attributions", "Compare Feature Ablation with Integrated Gradients for the same predictions", "Module 01 Notebook 02 (gradient methods on tabular data)"])

In [ ]:
import sys; sys.path.insert(0, '../../../../..')
from resources.notebook_style import apply_course_theme, learning_objectives, section_divider, callout, key_takeaways
from resources.graphics.plot_theme import apply_plot_theme

apply_course_theme()
apply_plot_theme()

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import urllib.request
import io

from captum.attr import FeatureAblation, IntegratedGradients

torch.manual_seed(42)
np.random.seed(42)
DEVICE = torch.device('cpu')  # Small tabular model — CPU is fine
print(f"Device: {DEVICE}")

## 1. Load UCI Wine Quality Dataset

The dataset contains 11 physicochemical properties of red wines, with quality scores from 0–10. We frame this as a binary classification: quality ≥ 6 = "good wine."

In [ ]:
# Load UCI Red Wine Quality dataset
WINE_URL = "https://archive.ics.uci.edu/ml/machine-learning-databases/wine-quality/winequality-red.csv"

try:
    with urllib.request.urlopen(WINE_URL, timeout=15) as resp:
        df = pd.read_csv(io.BytesIO(resp.read()), sep=';')
except Exception:
    # Fallback: generate synthetic data matching the wine dataset structure
    print("Could not download dataset — using synthetic fallback.")
    n = 1599
    np.random.seed(42)
    df = pd.DataFrame({
        'fixed acidity':          np.random.uniform(4.6, 15.9, n),
        'volatile acidity':        np.random.uniform(0.12, 1.58, n),
        'citric acid':             np.random.uniform(0.0, 1.0, n),
        'residual sugar':          np.random.uniform(1.2, 15.5, n),
        'chlorides':               np.random.uniform(0.012, 0.611, n),
        'free sulfur dioxide':     np.random.uniform(1.0, 72.0, n),
        'total sulfur dioxide':    np.random.uniform(6.0, 289.0, n),
        'density':                 np.random.uniform(0.990, 1.004, n),
        'pH':                      np.random.uniform(2.74, 4.01, n),
        'sulphates':               np.random.uniform(0.33, 2.0, n),
        'alcohol':                 np.random.uniform(8.4, 14.9, n),
        'quality':                 np.random.choice(range(3, 9), n)
    })

print(f"Dataset shape: {df.shape}")
print(f"Quality distribution:")
print(df['quality'].value_counts().sort_index().to_string())

# Feature names (shorter versions for plotting)
FEATURE_NAMES = [
    'fixed_acidity', 'volatile_acidity', 'citric_acid',
    'residual_sugar', 'chlorides', 'free_SO2',
    'total_SO2', 'density', 'pH', 'sulphates', 'alcohol'
]

# Binary target: quality >= 6 = good (1), quality < 6 = poor (0)
X = df.drop('quality', axis=1).values.astype(np.float32)
y = (df['quality'].values >= 6).astype(np.float32)

print(f"\nGood wine: {y.mean():.1%} | Poor wine: {(1-y).mean():.1%}")

In [ ]:
# Preprocess and train/test split
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y
)

X_train_t = torch.tensor(X_train, dtype=torch.float32)
X_test_t  = torch.tensor(X_test,  dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.float32)
y_test_t  = torch.tensor(y_test,  dtype=torch.float32)

# Training mean as baseline ("typical wine")
X_mean = torch.tensor(X_train.mean(axis=0), dtype=torch.float32).unsqueeze(0)

print(f"Train: {X_train.shape}, Test: {X_test.shape}")
print(f"Baseline (training mean): {X_mean.squeeze().numpy().round(3)}")


# Define MLP model
class WineQualityMLP(nn.Module):
    def __init__(self, n_features=11):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_features, 64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 2)  # 2 outputs for binary classification
        )

    def forward(self, x):
        return self.net(x)


model = WineQualityMLP().to(DEVICE)

# Train the model
optimizer = optim.Adam(model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()

print("Training WineQualityMLP...")
model.train()
for epoch in range(80):
    logits = model(X_train_t)
    loss = criterion(logits, y_train_t.long())
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

model.eval()
with torch.no_grad():
    test_logits = model(X_test_t)
    test_acc = (test_logits.argmax(1) == y_test_t.long()).float().mean().item()

print(f"Test accuracy: {test_acc:.1%}")

## 2. Feature Ablation: Per-Sample Attribution

We apply `FeatureAblation` to explain individual wine predictions. For each sample, we measure: "what happens to the model's prediction when we replace each feature with the training mean?"

A large positive attribution means: removing this feature (replacing with training mean) significantly reduces the "good wine" probability — so this feature is supporting the prediction.

In [ ]:
fa = FeatureAblation(model)

# Find two contrasting examples: a confidently-good and confidently-poor wine
with torch.no_grad():
    test_probs = torch.softmax(model(X_test_t), dim=1)
    good_probs = test_probs[:, 1]  # P(good wine)

# High-confidence good wine and high-confidence poor wine
good_idx = good_probs.argmax().item()
poor_idx = good_probs.argmin().item()

print(f"Confidently good wine — P(good) = {good_probs[good_idx]:.1%}")
print(f"Confidently poor wine — P(good) = {good_probs[poor_idx]:.1%}")

# Compute Feature Ablation for both samples
good_wine_tensor = X_test_t[good_idx].unsqueeze(0)  # (1, 11)
poor_wine_tensor = X_test_t[poor_idx].unsqueeze(0)  # (1, 11)

good_attr = fa.attribute(
    good_wine_tensor,
    baselines=X_mean,
    target=1  # Explain "good wine" class
)
poor_attr = fa.attribute(
    poor_wine_tensor,
    baselines=X_mean,
    target=1  # Explain "good wine" class
)

good_attrs = good_attr.squeeze().detach().numpy()
poor_attrs = poor_attr.squeeze().detach().numpy()

print(f"\nGood wine attribution sum: {good_attrs.sum():.4f}")
print(f"Poor wine attribution sum: {poor_attrs.sum():.4f}")

In [ ]:
# Visualize feature attributions for good vs poor wine
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle(
    'Feature Ablation: Confidently Good vs Poor Wine\n'
    '(Training mean baseline = "typical wine")',
    fontsize=12
)

for ax, attrs, title, prob_val in [
    (axes[0], good_attrs, f'Good Wine — P(good) = {good_probs[good_idx]:.1%}',
     good_probs[good_idx].item()),
    (axes[1], poor_attrs, f'Poor Wine — P(good) = {good_probs[poor_idx]:.1%}',
     good_probs[poor_idx].item()),
]:
    sorted_idx = np.argsort(attrs)
    colors = ['#2ecc71' if v >= 0 else '#e74c3c'
               for v in attrs[sorted_idx]]

    bars = ax.barh(
        [FEATURE_NAMES[i] for i in sorted_idx],
        attrs[sorted_idx],
        color=colors, alpha=0.85
    )
    ax.axvline(0, color='black', lw=0.8)
    ax.set_xlabel('Feature Ablation Attribution\n(positive = feature supports "good wine")')
    ax.set_title(title, fontsize=11)

    # Annotate bars with feature values (original unscaled)
    if ax is axes[0]:
        sample_orig = scaler.inverse_transform(
            X_test_t[good_idx].numpy().reshape(1, -1)
        ).squeeze()
    else:
        sample_orig = scaler.inverse_transform(
            X_test_t[poor_idx].numpy().reshape(1, -1)
        ).squeeze()

    for i, (bar, feat_idx) in enumerate(zip(bars, sorted_idx)):
        val = attrs[feat_idx]
        orig_val = sample_orig[feat_idx]
        ax.text(
            bar.get_width() + 0.0002, bar.get_y() + bar.get_height() / 2,
            f'{orig_val:.2f}', va='center', fontsize=7, color='gray'
        )

plt.tight_layout()
plt.show()

print("\nGreen bars: removing this feature (→ training mean) would hurt the prediction")
print("Red bars:   removing this feature would help — it was hurting the prediction")

## 3. Global Feature Importance

Aggregating local Feature Ablation attributions across many samples gives global feature importance — which features matter across the whole test set.

In [ ]:
# Compute Feature Ablation for all test samples
# (subset for speed — first 100 samples)
N_EXPLAIN = min(100, len(X_test_t))
all_attrs = []

print(f"Computing Feature Ablation for {N_EXPLAIN} test samples...")
for i in range(N_EXPLAIN):
    sample = X_test_t[i].unsqueeze(0)
    target = int(y_test_t[i].item())  # Explain the actual class

    attr = fa.attribute(
        sample,
        baselines=X_mean,
        target=target
    )
    all_attrs.append(attr.squeeze().detach().numpy())

    if (i + 1) % 25 == 0:
        print(f"  {i+1}/{N_EXPLAIN} done")

all_attrs = np.array(all_attrs)  # (N_EXPLAIN, 11)
print(f"Attribution matrix shape: {all_attrs.shape}")

# Global feature importance: mean absolute attribution
global_importance_mean = all_attrs.mean(axis=0)   # Signed mean
global_importance_abs  = np.abs(all_attrs).mean(axis=0)  # Mean absolute
global_importance_std  = all_attrs.std(axis=0)    # Variability

print("\nGlobal Feature Importance (mean absolute attribution):")
sorted_idx = global_importance_abs.argsort()[::-1]
for rank, feat_idx in enumerate(sorted_idx):
    print(f"  #{rank+1:2d} {FEATURE_NAMES[feat_idx]:20s}: "
          f"|mean|={global_importance_abs[feat_idx]:.5f}  "
          f"mean={global_importance_mean[feat_idx]:+.5f}")

In [ ]:
# Visualize global importance with error bars
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle(
    f'Global Feature Importance via Feature Ablation (N={N_EXPLAIN} test samples)',
    fontsize=12
)

# Mean absolute importance (ranked)
abs_idx = np.argsort(global_importance_abs)
axes[0].barh(
    [FEATURE_NAMES[i] for i in abs_idx],
    global_importance_abs[abs_idx],
    xerr=global_importance_std[abs_idx],
    color='steelblue', alpha=0.85,
    capsize=4
)
axes[0].set_xlabel('Mean |Attribution| ± std')
axes[0].set_title('Feature Importance (Mean Absolute Ablation)')

# Attribution distribution per feature (violin-like histogram using individual bars)
# Show distribution of signed attributions across samples
feature_order = np.argsort(global_importance_abs)[::-1][:8]  # Top 8
bp_data = [all_attrs[:, i] for i in feature_order]
bp = axes[1].boxplot(
    bp_data,
    vert=True,
    labels=[FEATURE_NAMES[i] for i in feature_order],
    patch_artist=True
)
for patch, median in zip(bp['boxes'], bp['medians']):
    patch.set_facecolor('steelblue')
    patch.set_alpha(0.7)
    median.set_color('red')
axes[1].axhline(0, color='black', lw=0.8, linestyle='--')
axes[1].set_xticklabels(
    [FEATURE_NAMES[i][:12] for i in feature_order],
    rotation=30, ha='right'
)
axes[1].set_ylabel('Signed Attribution')
axes[1].set_title('Attribution Distribution (Top 8 Features)')

plt.tight_layout()
plt.show()

most_important = FEATURE_NAMES[global_importance_abs.argmax()]
print(f"\nMost globally important feature: {most_important}")
print("(Expected: 'alcohol' is consistently the strongest predictor of wine quality)")

## 4. Feature Ablation vs. Integrated Gradients

Both methods should identify similar globally important features. We compare their rankings.

In [ ]:
from scipy.stats import spearmanr

ig = IntegratedGradients(model)

# Compute IG for the same N_EXPLAIN samples
ig_attrs = []
print(f"Computing IG for {N_EXPLAIN} test samples...")
for i in range(N_EXPLAIN):
    sample = X_test_t[i].unsqueeze(0).requires_grad_(True)
    target = int(y_test_t[i].item())

    ig_attr, delta = ig.attribute(
        sample, baselines=X_mean,
        target=target, n_steps=50,
        return_convergence_delta=True
    )
    ig_attrs.append(ig_attr.squeeze().detach().numpy())

    if (i + 1) % 25 == 0:
        print(f"  {i+1}/{N_EXPLAIN} done")

ig_attrs = np.array(ig_attrs)  # (N_EXPLAIN, 11)

# Global importance for IG
ig_global_abs = np.abs(ig_attrs).mean(axis=0)

# Rank correlation between FA and IG global importances
rho, pval = spearmanr(global_importance_abs, ig_global_abs)
print(f"\nSpearman ρ (FA vs IG global importance): {rho:.4f}")

# Comparison figure
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle(
    f'Feature Ablation vs. Integrated Gradients — Global Importance\n'
    f'Spearman ρ = {rho:.3f}',
    fontsize=12
)

# Side-by-side bar chart
x = np.arange(len(FEATURE_NAMES))
w = 0.38

# Sort by IG importance
ig_sort_idx = np.argsort(ig_global_abs)

axes[0].barh([FEATURE_NAMES[i] for i in ig_sort_idx],
              ig_global_abs[ig_sort_idx],
              height=w, label='Integrated Gradients', color='#3498db', alpha=0.8)
axes[0].barh([FEATURE_NAMES[i] for i in ig_sort_idx],
              global_importance_abs[ig_sort_idx],
              height=w, left=0,
              label='Feature Ablation', color='#e74c3c', alpha=0.6)
axes[0].set_xlabel('Mean |Attribution|')
axes[0].set_title('Global Importance Comparison\n(ordered by IG rank)')
axes[0].legend()

# Scatter: FA vs IG per feature
axes[1].scatter(ig_global_abs, global_importance_abs,
                 s=100, c='navy', alpha=0.8)
for i, name in enumerate(FEATURE_NAMES):
    axes[1].annotate(name[:10],
                      (ig_global_abs[i], global_importance_abs[i]),
                      fontsize=7, ha='left', va='bottom')
axes[1].set_xlabel('IG Mean |Attribution|')
axes[1].set_ylabel('Feature Ablation Mean |Attribution|')
axes[1].set_title(f'FA vs IG: Per-Feature Correlation\nρ = {rho:.3f}')

plt.tight_layout()
plt.show()

# Auto-check
assert all_attrs.shape == (N_EXPLAIN, 11), "Attribution matrix shape incorrect"
assert ig_attrs.shape == (N_EXPLAIN, 11), "IG attribution matrix shape incorrect"
print("\nAll checks passed.")

## Summary

### What You Learned

1. **Feature Ablation** measures the effect of replacing each feature with a baseline value — a direct causal test
2. **Training mean as baseline** represents a "typical wine" reference, so attribution measures deviation from typical
3. **Per-sample attribution** explains individual predictions; **global importance** aggregates across the dataset
4. **Alcohol** is typically the most important predictor of wine quality — consistent across FA and IG
5. **FA vs. IG agreement** (high Spearman ρ) validates both methods are capturing genuine feature importance

### Key API
```python
from captum.attr import FeatureAblation

fa = FeatureAblation(model)
attr = fa.attribute(
    input_tensor,          # (1, n_features)
    baselines=X_mean,      # Training mean
    target=1               # Target class
)
# attr: (1, n_features) — one importance score per feature
```

### What's Next

**Notebook 03:** Shapley Value Sampling — compare with IG and Feature Ablation, analyze convergence, and visualize the game-theoretic attribution.